# Investigate & Validate a "Trend-Loyalty" Market Regime

**Question.** A *regime* here = **how loyal the market is to its current trend**
(trend continuation). This notebook reaches one of three verdicts, with numbers:

1. **Validated *with* zigzag** — trend features separate the future *and* chop adds lift on top.
2. **Validated *without* zigzag** — trend features separate the future, but chop adds nothing once trend is held fixed.
3. **Not validated** — neither separates the future beyond the null.

A regime exists **only to the degree that conditioning on the state shifts the
forward-outcome distribution away from the base rate, with separation, and that
this survives out-of-sample.**

### Non-negotiable principles (enforced throughout)
1. **No forward leak** — every feature is built on the 1440 look-back only; the zigzag is rebuilt from scratch on the 1440 window per snapshot; the 240 is outcome-only.
2. **Lift, not absolute probability** — always vs the unconditional base rate and vs the complement.
3. **Snapshots are not independent** — headline CIs use non-overlapping stride-240 outcomes or block bootstrap (block >= 240). The naive-vs-corrected gap is shown once.
4. **Walk-forward, never shuffle** — define on the past, check on the future; stability over time is part of the validation.
5. **Beat the null** — surrogates that preserve return distribution + volatility clustering but destroy regime structure must collapse the lift.

Follows `idea_look_back_look_ahead.md` (windowing, exclusive boundaries, column reference) and
`idea_normalize_based_on_last_price_clip.md` (k=100, hard clip [-1,1], anchor = last look-back vwap).

In [ ]:
%pip install numpy pandas pyarrow requests huggingface_hub numba scipy scikit-learn plotly

In [ ]:
import os
import sys

# Branch-aware clone so local project imports resolve at runtime (rules.md).
REPO_URL   = "https://github.com/payamdav/pycrypto.git"
REPO_NAME  = "pycrypto"
BRANCH     = "claude/upbeat-heisenberg-7212jv"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

# When the notebook is run from inside the repo checkout, this no-ops harmlessly.
print("repo path on sys.path:", REPO_PATH)

In [ ]:
import io
import re
import time
import math
import warnings
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
import pyarrow.parquet as pq
import numba as nb
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
np.random.seed(7)

print("imports ok")

## Config (defaults from the brief, section 7)

In [ ]:
# ----------------------------- CONFIG -----------------------------
CFG = dict(
    assets       = ["btcusdt", "ethusdt", "trumpusdt", "vineusdt",
                    "adausdt", "xrpusdt", "dogeusdt"],
    start        = "2024-01-01",          # full available range; missing months skipped
    end          = "2026-05-31",          # ~ up to current coverage
    price_col    = "vwap",
    look_back    = 1440,
    look_ahead   = 240,
    tf           = 1,                     # minutes per candle

    # normalization (idea_normalize_based_on_last_price_clip.md)
    k            = 100.0,                  # ratio R = 0.01  -> +-1%
    clip_min     = -1.0,
    clip_max     = 1.0,

    zz_deltas    = (2.0, 1.0, 0.5, 0.2),  # on the clipped series; prune dead deltas
    delta_age    = 0.5,                    # trend_age knot delta
    reg_windows  = (1440, 720, 360, 180),
    H            = 720,                    # trend-defining horizon (test all four)

    stride       = 240,                   # non-overlapping forward windows (headline)
    stride_dense = 60,                    # dense heatmaps -> block bootstrap CIs

    tau          = 0.0,                   # deadband for cont_bin
    barrier_b    = 0.01,                  # triple-barrier band (= clip band)

    wf_folds     = 6,
    n_qbins      = 8,                     # quantile bins per axis
    min_cell_N   = 50,                    # mask heatmap cells below this
    n_surrogate  = 200,                   # surrogate null draws
    block_len    = 240,                   # block bootstrap block length
    embargo      = 240,                   # purge/embargo minutes at split boundary

    # to keep the notebook runnable end-to-end, cap snapshots per asset.
    # set to None to use every stride-240 snapshot in the period.
    max_snaps_per_asset = None,
)

LB  = CFG["look_back"]
LA  = CFG["look_ahead"]
print("config set. look_back=%d look_ahead=%d stride=%d" % (LB, LA, CFG["stride"]))

## Data retrieval

Minute candles from the HuggingFace `payamdavaee/candles` dataset
(`huggingface_candles.md`). We load whole monthly files per asset, keep `ts` and
`vwap`, and rely on **exclusive boundaries**: every snapshot anchored inside the
period has a full 1440 look-back and a full 240 look-ahead because we load the
entire contiguous history available for the asset and only *anchor* on rows that
have both windows complete.

In [ ]:
from huggingface_hub import HfApi

REPO = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"
_api = HfApi()
_ALL_FILES = _api.list_repo_files("payamdavaee/candles", repo_type="dataset")

def list_asset_months(asset):
    pat = re.compile(rf"^{asset}/{asset}-1m-(\d{{4}})-(\d{{2}})\.parquet$")
    out = []
    for f in _ALL_FILES:
        m = pat.match(f)
        if m:
            out.append((int(m.group(1)), int(m.group(2)), f))
    return sorted(out)

def load_asset_vwap(asset, start, end):
    """Return a sorted (ts_ms, vwap) DataFrame for [start, end] inclusive months."""
    s_dt = datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
    e_dt = datetime.fromisoformat(end).replace(tzinfo=timezone.utc)
    frames = []
    for (y, mo, f) in list_asset_months(asset):
        first = datetime(y, mo, 1, tzinfo=timezone.utc)
        if first > e_dt + timedelta(days=31):
            continue
        if first < s_dt - timedelta(days=31):
            continue
        url = f"{REPO}/{f}"
        for attempt in range(4):
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                tbl = pq.read_table(io.BytesIO(resp.content), columns=["ts", "vwap"])
                frames.append(tbl.to_pandas())
                break
            except Exception as ex:
                if attempt == 3:
                    print(f"  WARN failed {f}: {ex}")
                else:
                    time.sleep(2 ** attempt)
    if not frames:
        return pd.DataFrame(columns=["ts", "vwap"])
    df = pd.concat(frames).sort_values("ts").reset_index(drop=True)
    df = df.drop_duplicates("ts").reset_index(drop=True)
    df["ts"] = df["ts"].astype("int64")
    # vwap can be NaN/0 on empty candles -> forward fill to keep a continuous series
    df["vwap"] = df["vwap"].replace(0.0, np.nan).ffill().bfill()
    return df

print("loader ready")

## Core kernels (numba)

* **Zigzag knot count** — scalar loop over the 1440 clipped series (no leak; rebuilt per snapshot). `knots - 2` = internal reversals.
* **Last-knot index** for `trend_age`.
* **OLS slope & R²** over a series vs the time index (used for each regression window, on both clipped and unclipped normalized series).
* **Triple-barrier / MFE / MAE** over the forward 240.

In [ ]:
@nb.njit(cache=True)
def zigzag_count_and_lastknot(x, delta):
    """Count zigzag knots on series x with threshold delta.
    Returns (n_knots, last_knot_index). knots include the two endpoints implicitly
    via the standard zigzag: we count confirmed pivots plus the running endpoint.
    last_knot_index = index of the most recent confirmed pivot (or 0)."""
    n = x.shape[0]
    if n == 0:
        return 0, 0
    # running extreme since last pivot
    last_pivot_val = x[0]
    last_pivot_idx = 0
    cur_ext_val = x[0]
    cur_ext_idx = 0
    direction = 0          # 0 unknown, +1 up leg, -1 down leg
    n_pivots = 1           # the first point is a knot
    last_knot_idx = 0
    for i in range(1, n):
        v = x[i]
        if direction == 0:
            if v - last_pivot_val >= delta:
                direction = 1
                cur_ext_val = v; cur_ext_idx = i
            elif last_pivot_val - v >= delta:
                direction = -1
                cur_ext_val = v; cur_ext_idx = i
            else:
                if v > cur_ext_val:
                    cur_ext_val = v; cur_ext_idx = i
                if v < last_pivot_val:
                    last_pivot_val = v; last_pivot_idx = i  # drift the anchor
        elif direction == 1:
            if v > cur_ext_val:
                cur_ext_val = v; cur_ext_idx = i
            elif cur_ext_val - v >= delta:
                # confirm pivot at cur_ext
                n_pivots += 1
                last_knot_idx = cur_ext_idx
                last_pivot_val = cur_ext_val; last_pivot_idx = cur_ext_idx
                direction = -1
                cur_ext_val = v; cur_ext_idx = i
        else:  # direction == -1
            if v < cur_ext_val:
                cur_ext_val = v; cur_ext_idx = i
            elif v - cur_ext_val >= delta:
                n_pivots += 1
                last_knot_idx = cur_ext_idx
                last_pivot_val = cur_ext_val; last_pivot_idx = cur_ext_idx
                direction = 1
                cur_ext_val = v; cur_ext_idx = i
    # add the final running extreme as the closing knot
    n_knots = n_pivots + 1
    return n_knots, last_knot_idx


@nb.njit(cache=True)
def ols_slope_r2(y):
    """OLS of y on integer time index 0..n-1. Returns (slope, r2)."""
    n = y.shape[0]
    if n < 2:
        return 0.0, 0.0
    sx = 0.0; sy = 0.0
    for i in range(n):
        sx += i; sy += y[i]
    mx = sx / n; my = sy / n
    sxx = 0.0; sxy = 0.0; syy = 0.0
    for i in range(n):
        dx = i - mx; dy = y[i] - my
        sxx += dx * dx; sxy += dx * dy; syy += dy * dy
    if sxx == 0.0:
        return 0.0, 0.0
    slope = sxy / sxx
    if syy == 0.0:
        return slope, 0.0
    r = sxy / math.sqrt(sxx * syy)
    return slope, r * r


@nb.njit(cache=True)
def triple_barrier(fwd_ret, trend_dir, b):
    """fwd_ret: array of (vwap[t+k]/price_l - 1) for k=1..LA.
    Returns (barrier_hit, MFE, MAE) in the trend_dir frame."""
    n = fwd_ret.shape[0]
    mfe = -1e18; mae = 1e18
    hit = 0
    for k in range(n):
        d = trend_dir * fwd_ret[k]
        if d > mfe: mfe = d
        if d < mae: mae = d
        if hit == 0:
            if d >= b:
                hit = 1
            elif d <= -b:
                hit = -1
    if n == 0:
        mfe = 0.0; mae = 0.0
    return hit, mfe, mae

print("kernels compiled lazily on first call")

## A. Build & store — one row per snapshot

For each asset we slide an anchor at `stride` over rows that have a full
1440 look-back behind and a full 240 look-ahead ahead. Per snapshot:

* normalize the 1440 vwap window (clipped & unclipped),
* zigzag knot counts for each delta on the **clipped** series + `trend_age`,
* OLS slope & R² over `{1440,720,360,180}` on **both** clipped and unclipped series,
* forward outcomes on the 240 (`fwd_ret_240`, `cont_mag`, `cont_bin`, barrier/MFE/MAE).

All features are look-back-only; outcomes are look-ahead-only. No leak.

In [ ]:
def normalize_lb(vwap_window, k, cmin, cmax):
    """price_l = last look-back vwap; raw = k*(vwap/price_l - 1); clipped to [cmin,cmax]."""
    price_l = vwap_window[-1]
    raw = k * (vwap_window / price_l - 1.0)
    clipped = np.clip(raw, cmin, cmax)
    return raw, clipped, price_l


def build_asset_snapshots(asset, df, cfg):
    LB, LA = cfg["look_back"], cfg["look_ahead"]
    stride = cfg["stride"]
    deltas = cfg["zz_deltas"]
    wins   = cfg["reg_windows"]
    k, cmin, cmax = cfg["k"], cfg["clip_min"], cfg["clip_max"]

    ts   = df["ts"].to_numpy(np.int64)
    vw   = df["vwap"].to_numpy(np.float64)
    N    = len(vw)

    first_anchor = LB - 1
    last_anchor  = N - LA - 1
    if last_anchor < first_anchor:
        return pd.DataFrame()

    anchors = list(range(first_anchor, last_anchor + 1, stride))
    if cfg["max_snaps_per_asset"]:
        anchors = anchors[: cfg["max_snaps_per_asset"]]

    rows = []
    for i in anchors:
        lb_vw = vw[i - LB + 1 : i + 1]                 # (1440,)
        la_vw = vw[i + 1 : i + 1 + LA]                 # (240,)
        if lb_vw[-1] <= 0 or not np.isfinite(lb_vw[-1]):
            continue

        raw_n, clip_n, price_l = normalize_lb(lb_vw, k, cmin, cmax)

        rec = {"asset": asset, "ts": int(ts[i])}

        # 3a zigzag on clipped series
        for d in deltas:
            nk, _ = zigzag_count_and_lastknot(clip_n, float(d))
            rec[f"knot_{d}"] = nk - 2          # internal reversals
        # trend age from delta_age knots
        _, lastknot = zigzag_count_and_lastknot(clip_n, float(cfg["delta_age"]))
        rec["trend_age"] = (LB - 1) - lastknot  # minutes since last knot

        # 3b slope & r2 on both clipped and unclipped, per window (right-aligned)
        for w in wins:
            seg_c = clip_n[-w:]
            seg_u = raw_n[-w:]
            sc, r2c = ols_slope_r2(seg_c)
            su, r2u = ols_slope_r2(seg_u)
            rec[f"slope_c_{w}"] = sc; rec[f"r2_c_{w}"] = r2c
            rec[f"slope_u_{w}"] = su; rec[f"r2_u_{w}"] = r2u

        # forward outcomes (look-ahead only)
        fwd_ret_full = la_vw / price_l - 1.0           # (240,)
        rec["fwd_ret_240"] = float(la_vw[-1] / price_l - 1.0)
        rec["fwd_path"] = fwd_ret_full                  # keep for barrier (dropped before save)
        rows.append(rec)

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)

print("snapshot builder ready")

In [ ]:
# Build all assets. (Network + compile heavy on first call.)
t0 = time.time()
frames = []
for a in CFG["assets"]:
    print(f"== {a} ==")
    df = load_asset_vwap(a, CFG["start"], CFG["end"])
    if df.empty:
        print("  no data, skipping"); continue
    print(f"  candles: {len(df):,}  range {pd.to_datetime(df.ts.iloc[0],unit='ms')} -> {pd.to_datetime(df.ts.iloc[-1],unit='ms')}")
    snaps = build_asset_snapshots(a, df, CFG)
    print(f"  snapshots: {len(snaps):,}")
    if not snaps.empty:
        frames.append(snaps)
print(f"total build time: {time.time()-t0:.1f}s")
snap = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print("pooled snapshots:", len(snap))

### Derived axes (section 5) and final outcomes (section 4)

`trend_dir` is set by `sign(slope_H)` at the trend-defining horizon `H` (default 720).
We default to the **clipped** slope for direction but keep both available. The
forward outcomes (`cont_mag`, `cont_bin`, barrier/MFE/MAE) depend on `trend_dir`,
so they are finalized here — still with zero leak (direction is a look-back
feature; outcome uses only the forward path).

In [ ]:
def finalize(snap, cfg, H=None, slope_kind="c"):
    H = H or cfg["H"]
    df = snap.copy()
    wins = cfg["reg_windows"]

    # direction per horizon (use chosen slope kind for direction)
    for w in wins:
        df[f"dir_{w}"] = np.sign(df[f"slope_{slope_kind}_{w}"]).astype(int)
    df["trend_dir"] = df[f"dir_{H}"].replace(0, 1)   # tie -> +1

    # trend descriptors at H
    df["r2_H"]    = df[f"r2_{slope_kind}_{H}"]
    df["slope_H"] = df[f"slope_{slope_kind}_{H}"]
    df["mag_H"]   = df["slope_H"].abs()
    df["drift_H"] = df["slope_H"] * (H - 1)          # ~ % move modeled over window
    df["align"]   = df[[f"dir_{w}" for w in wins]].sum(axis=1).abs() / len(wins)

    # chop scalars
    df["chop"]        = df[f"knot_{cfg['chop_scalar_delta']}"]
    df["chop_ratio"]  = df["knot_0.2"] / np.maximum(df["knot_1.0"], 1)

    # forward outcomes finalized with trend_dir
    td  = df["trend_dir"].to_numpy()
    fr  = df["fwd_ret_240"].to_numpy()
    df["cont_mag"] = td * fr
    df["cont_bin"] = (df["cont_mag"] > cfg["tau"]).astype(int)

    # path outcomes via triple-barrier
    bh = np.zeros(len(df), np.int64); mfe = np.zeros(len(df)); mae = np.zeros(len(df))
    paths = df["fwd_path"].to_numpy()
    for j in range(len(df)):
        h, f_, a_ = triple_barrier(paths[j], int(td[j]), cfg["barrier_b"])
        bh[j] = h; mfe[j] = f_; mae[j] = a_
    df["barrier_hit"] = bh
    df["MFE"] = mfe; df["MAE"] = mae
    return df

CFG["chop_scalar_delta"] = 0.5
snap_f = finalize(snap, CFG)
print("finalized rows:", len(snap_f))
print(snap_f[["asset","ts","trend_dir","r2_H","mag_H","chop","trend_age",
              "fwd_ret_240","cont_mag","cont_bin","barrier_hit"]].head())

In [ ]:
# Persist (parquet per-asset + pooled). Drop the heavy path column for storage.
os.makedirs("trend_loyalty_out", exist_ok=True)
save_cols = [c for c in snap_f.columns if c != "fwd_path"]
snap_save = snap_f[save_cols]
snap_save.to_parquet("trend_loyalty_out/snapshots_pooled.parquet", index=False)
for a, g in snap_save.groupby("asset"):
    g.to_parquet(f"trend_loyalty_out/snapshots_{a}.parquet", index=False)

# coverage report
print("=== coverage ===")
cov = snap_save.assign(date=pd.to_datetime(snap_save.ts, unit="ms")).groupby("asset")["date"].agg(["count","min","max"])
print(cov)
print("\n=== delta-variance report (3a): prune near-constant deltas ===")
for d in CFG["zz_deltas"]:
    col = snap_save[f"knot_{d}"]
    print(f"  knot_{d}: mean={col.mean():.2f} std={col.std():.2f} "
          f"min={col.min()} max={col.max()} "
          f"{'<-- NEAR-CONSTANT, candidate to drop' if col.std() < 0.5 else ''}")

### Overlap-correction helpers

Headline stats use the **stride-240 non-overlapping** snapshots directly (binomial
CI valid). For dense sampling we provide a **block bootstrap** (block >= 240) so
CIs respect serial dependence. We also expose a naive binomial CI to display the
mirage gap once.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    den = 1 + z*z/n
    cen = (p + z*z/(2*n)) / den
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return (cen-half, cen+half)

def block_bootstrap_mean(x, block, n_boot=400, agg=np.mean, seed=0):
    """Circular block bootstrap CI for agg(x)."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    n = len(x)
    if n == 0: return (np.nan, np.nan, np.nan)
    nb_blocks = int(np.ceil(n / block))
    stats_ = np.empty(n_boot)
    idx_base = np.arange(block)
    for b in range(n_boot):
        starts = rng.integers(0, n, size=nb_blocks)
        idx = (starts[:, None] + idx_base[None, :]).ravel() % n
        idx = idx[:n]
        stats_[b] = agg(x[idx])
    return (agg(x), np.percentile(stats_, 2.5), np.percentile(stats_, 97.5))

print("CI helpers ready")

## B. Base rates & 1-D marginals

Unconditional `cont_bin` base rate and `cont_mag` distribution (overall, per asset,
per epoch), then each axis alone vs continuation — **looking explicitly for
non-monotonicity** (exhaustion at extreme R², U-shaped chop).

In [ ]:
N = len(snap_f)
base_rate = snap_f["cont_bin"].mean()
print(f"Unconditional continuation base rate: {base_rate:.4f}  (N={N:,})")
print(f"cont_mag: mean={snap_f.cont_mag.mean():+.5f}  median={snap_f.cont_mag.median():+.5f}  std={snap_f.cont_mag.std():.5f}")
lo, hi = wilson_ci(snap_f.cont_bin.sum(), N)
print(f"base-rate 95% Wilson CI: [{lo:.4f}, {hi:.4f}]")

print("\nPer asset:")
pa = snap_f.groupby("asset").agg(n=("cont_bin","size"),
                                 cont_rate=("cont_bin","mean"),
                                 mean_cont_mag=("cont_mag","mean"))
print(pa)

snap_f["epoch"] = pd.to_datetime(snap_f.ts, unit="ms").dt.to_period("Q").astype(str)
print("\nPer epoch (quarter):")
print(snap_f.groupby("epoch").agg(n=("cont_bin","size"), cont_rate=("cont_bin","mean")))

In [ ]:
# 1-D marginals with overlap-corrected (stride-240 -> binomial-valid) CIs.
def qbin(s, nq):
    try:
        return pd.qcut(s.rank(method="first"), nq, labels=False)
    except Exception:
        return pd.cut(s, nq, labels=False)

def marginal_table(df, axis, nq):
    d = df.copy()
    d["bin"] = qbin(d[axis], nq)
    rows = []
    for b, g in d.groupby("bin"):
        lo, hi = wilson_ci(g.cont_bin.sum(), len(g))
        rows.append(dict(bin=int(b), axis_mid=g[axis].median(), n=len(g),
                         cont_rate=g.cont_bin.mean(), ci_lo=lo, ci_hi=hi,
                         mean_cont_mag=g.cont_mag.mean()))
    return pd.DataFrame(rows)

axes_1d = ["r2_H", "mag_H", "chop", "chop_ratio", "trend_age", "align"]
fig = make_subplots(rows=2, cols=3, subplot_titles=axes_1d)
for idx, ax in enumerate(axes_1d):
    t = marginal_table(snap_f, ax, CFG["n_qbins"])
    r, c = idx//3+1, idx%3+1
    fig.add_trace(go.Scatter(x=t.axis_mid, y=t.cont_rate, mode="lines+markers",
                  error_y=dict(type="data", symmetric=False,
                               array=t.ci_hi-t.cont_rate, arrayminus=t.cont_rate-t.ci_lo),
                  name=ax, showlegend=False), row=r, col=c)
    fig.add_hline(y=base_rate, line_dash="dot", line_color="gray", row=r, col=c)
fig.update_layout(height=640, width=1050, title_text="B. Continuation rate vs each axis (dotted = base rate; bars = 95% CI)")
fig.show()
for ax in axes_1d:
    print(f"\n--- {ax} ---"); print(marginal_table(snap_f, ax, CFG["n_qbins"]).to_string(index=False))

**Read the shapes above:** a monotone rise in R² would support "cleaner trend ->
more loyal"; a *rise-then-fall* signals exhaustion/blow-off at extreme cleanliness.
For chop, a **U-shape** (low continuation at both very-low and very-high chop) would
say orderly pullbacks are healthy but violent chop is not. The verdict cell uses
the rank correlations computed next, not eyeballing.

## C. Headline ablation — with vs without zigzag (the crux)

1. **Trend-only separation** — does trend strength alone tell us about the future? (Spearman + AUC of trend strength -> `cont_bin`.)
2. **Conditional zigzag test** — within each trend-strength band, does chop separate continuation *holding trend fixed*?
3. **Rigorous increment** — walk-forward OOS AUC/Brier/logloss/R² for trend-only vs trend+chop, conditional MI, and a **permutation null** (shuffle chop within trend bins).
4. **Verdict.**

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, r2_score

def spearman(a, b):
    return stats.spearmanr(a, b, nan_policy="omit").correlation

# C1 trend-only separation. Trend strength = signed cleanliness*magnitude proxy:
snap_f["trend_strength"] = snap_f["r2_H"] * snap_f["mag_H"]   # cleanliness x magnitude
auc_trend = roc_auc_score(snap_f.cont_bin, snap_f.trend_strength)
rho_r2    = spearman(snap_f.r2_H,   snap_f.cont_mag)
rho_mag   = spearman(snap_f.mag_H,  snap_f.cont_mag)
rho_str   = spearman(snap_f.trend_strength, snap_f.cont_mag)
print("C1 trend-only separation")
print(f"  AUC(trend_strength -> cont_bin)        = {auc_trend:.4f}")
print(f"  Spearman(r2_H,  cont_mag)              = {rho_r2:+.4f}")
print(f"  Spearman(mag_H, cont_mag)              = {rho_mag:+.4f}")
print(f"  Spearman(trend_strength, cont_mag)     = {rho_str:+.4f}")
print(f"  (base rate {base_rate:.4f}; AUC=0.5 means trend tells us nothing)")

In [ ]:
# C2 conditional zigzag test: within each trend-strength band, does chop separate?
TS_BANDS = CFG["n_qbins"]
snap_f["ts_band"] = qbin(snap_f["trend_strength"], TS_BANDS)
rows = []
for tb, g in snap_f.groupby("ts_band"):
    if len(g) < 50: continue
    g = g.copy()
    g["chop_band"] = qbin(g["chop"], 3)   # low/mid/high chop within the trend band
    sub = g.groupby("chop_band").agg(n=("cont_bin","size"), cont_rate=("cont_bin","mean"),
                                     mean_cont_mag=("cont_mag","mean"), chop_med=("chop","median"))
    # separation across chop holding trend fixed
    rho = spearman(g.chop, g.cont_mag)
    spread = sub.cont_rate.max() - sub.cont_rate.min()
    rows.append(dict(ts_band=int(tb), n=len(g), trend_str_med=g.trend_strength.median(),
                     base=g.cont_bin.mean(), chop_spread=spread, rho_chop_cont=rho))
cond = pd.DataFrame(rows)
print("C2 conditional chop separation (within trend bands)")
if cond.empty:
    print("  (no trend band met the min-size threshold; lower n_qbins or check sample size)")
    mean_abs_rho_chop = 0.0
else:
    print(cond.to_string(index=False))
    mean_abs_rho_chop = cond.rho_chop_cont.abs().mean()
    print(f"\n  mean |rho(chop,cont_mag) | trend| = {mean_abs_rho_chop:.4f}")
    print(f"  mean continuation-rate spread across chop (trend fixed) = {cond.chop_spread.mean():.4f}")
    print("  -> if these are ~0, chop is a momentum signal in a costume.")

In [ ]:
# C3a walk-forward OOS increment: trend-only vs trend+chop predictors.
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

snap_f = snap_f.sort_values("ts").reset_index(drop=True)
trend_feats = ["r2_H","mag_H","align"] + [f"dir_{w}" for w in CFG["reg_windows"]]
chop_feats  = ["chop","chop_ratio","knot_0.2","trend_age"]

def wf_eval(df, feats_a, feats_b, folds, embargo_min):
    # expanding walk-forward over time, purge+embargo at each split
    ts = df.ts.to_numpy()
    n = len(df)
    bounds = np.linspace(0, n, folds+1).astype(int)
    res = []
    for f in range(1, folds):
        tr_end = bounds[f]
        te_beg = bounds[f]; te_end = bounds[f+1]
        # embargo: drop train rows whose forward window overlaps test start
        split_ts = ts[te_beg]
        tr_mask = ts < (split_ts - embargo_min*60_000 - LA*60_000)
        tr = df[tr_mask]
        te = df.iloc[te_beg:te_end]
        if len(tr) < 200 or len(te) < 50: continue
        out = dict(fold=f, n_tr=len(tr), n_te=len(te))
        for tag, feats in (("trend", feats_a), ("trend+chop", feats_b)):
            sc = StandardScaler().fit(tr[feats])
            Xtr, Xte = sc.transform(tr[feats]), sc.transform(te[feats])
            # classification cont_bin
            clf = LogisticRegression(max_iter=500).fit(Xtr, tr.cont_bin)
            p = clf.predict_proba(Xte)[:,1]
            out[f"auc_{tag}"]   = roc_auc_score(te.cont_bin, p) if te.cont_bin.nunique()>1 else np.nan
            out[f"brier_{tag}"] = brier_score_loss(te.cont_bin, p)
            out[f"logloss_{tag}"] = log_loss(te.cont_bin, np.clip(p,1e-6,1-1e-6), labels=[0,1])
            # regression cont_mag (OOS R2)
            reg = LinearRegression().fit(Xtr, tr.cont_mag)
            out[f"r2_{tag}"] = r2_score(te.cont_mag, reg.predict(Xte))
        res.append(out)
    return pd.DataFrame(res)

wf = wf_eval(snap_f, trend_feats, trend_feats+chop_feats, CFG["wf_folds"], CFG["embargo"])
print("C3a walk-forward OOS (trend-only vs trend+chop)")
wf_delta = {}
if wf.empty:
    print("  (no fold met the min train/test size; reduce wf_folds or increase data)")
    for m in ["auc","brier","logloss","r2"]:
        wf_delta[m] = float("nan")
else:
    print(wf.to_string(index=False))
    print("\nMean deltas (trend+chop minus trend-only):")
    for m in ["auc","brier","logloss","r2"]:
        d = (wf[f"{m}_trend+chop"] - wf[f"{m}_trend"]).mean()
        wf_delta[m] = d
        print(f"  delta_{m} = {d:+.5f}   ({'higher better' if m in ('auc','r2') else 'lower better'})")

In [ ]:
# C3b conditional mutual information I(cont ; chop | trend_bin)  [discretized]
def entropy(counts):
    p = counts / counts.sum()
    p = p[p>0]
    return -(p*np.log(p)).sum()

def cond_mi(df, x, y, cond, nbins=5):
    d = df.copy()
    d["xb"] = qbin(d[x], nbins); d["yb"] = d[y]   # y is already binary cont_bin
    d["cb"] = qbin(d[cond], nbins)
    mi = 0.0; ntot = len(d)
    for cv, g in d.groupby("cb"):
        w = len(g)/ntot
        # MI(xb, yb) within this cond bin
        ct = pd.crosstab(g.xb, g.yb).to_numpy(float)
        if ct.sum()==0: continue
        hx = entropy(ct.sum(1)); hy = entropy(ct.sum(0)); hxy = entropy(ct.ravel())
        mi += w * max(hx+hy-hxy, 0.0)
    return mi

cmi = cond_mi(snap_f, "chop", "cont_bin", "trend_strength", nbins=CFG["n_qbins"])
mi_uncond = cond_mi(snap_f.assign(_c=0), "chop", "cont_bin", "_c", nbins=CFG["n_qbins"])
print(f"C3b  I(cont ; chop | trend) = {cmi:.5f} nats")
print(f"     I(cont ; chop)         = {mi_uncond:.5f} nats  (unconditional, for scale)")

In [ ]:
# C3c permutation null: shuffle chop WITHIN trend bins, recompute conditional separation.
def conditional_stat(df):
    s = []
    for tb, g in df.groupby("ts_band"):
        if len(g) < 50: continue
        s.append(abs(spearman(g.chop, g.cont_mag)))
    return np.mean(s) if s else 0.0

obs = conditional_stat(snap_f)
rng = np.random.default_rng(11)
null = np.empty(300)
for i in range(300):
    d = snap_f.copy()
    d["chop"] = d.groupby("ts_band")["chop"].transform(lambda x: rng.permutation(x.values))
    null[i] = conditional_stat(d)
pval = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f"C3c permutation null (shuffle chop within trend bands)")
print(f"   observed mean|rho(chop,cont)| trend| = {obs:.5f}")
print(f"   null mean = {null.mean():.5f}  null 95th pct = {np.percentile(null,95):.5f}")
print(f"   p-value = {pval:.4f}  ({'chop beats the null' if pval<0.05 else 'chop does NOT beat the null'})")

## D. The 2-D continuation surface — let regimes emerge

Grid the plane; per cell compute `N`, continuation rate (+CI), mean `cont_mag`, and
**dispersion** (std of `fwd_ret_240`). Mask low-N cells. We **fold by trend
direction** (up vs down separately) and check whether the high region is the
"strong-trend/low-chop corner" or an **interior** peak, then facet by trend-age.

In [ ]:
def surface(df, ax_x, ax_y, nq, minN):
    d = df.copy()
    d["bx"] = qbin(d[ax_x], nq); d["by"] = qbin(d[ax_y], nq)
    g = d.groupby(["by","bx"])
    rate = g.cont_bin.mean().unstack()
    n    = g.cont_bin.size().unstack()
    mag  = g.cont_mag.mean().unstack()
    disp = g.fwd_ret_240.std().unstack()
    mask = n < minN
    return rate.mask(mask), n, mag.mask(mask), disp.mask(mask)

def show_surface(df, ax_x, ax_y, title):
    rate,n,mag,disp = surface(df, ax_x, ax_y, CFG["n_qbins"], CFG["min_cell_N"])
    fig = make_subplots(rows=1, cols=4, subplot_titles=["cont rate","mean cont_mag","N","dispersion"])
    for j,(M,cs) in enumerate([(rate,"RdYlGn"),(mag,"RdYlGn"),(n,"Viridis"),(disp,"Magma")]):
        fig.add_trace(go.Heatmap(z=M.values, colorscale=cs, showscale=False), row=1, col=j+1)
    fig.update_layout(height=320, width=1200, title_text=title)
    fig.show()
    return rate, n

print("D primary surface: r2_H (cleanliness) x chop, folded by trend direction")
for dname, sub in [("UP trend", snap_f[snap_f.trend_dir>0]),
                   ("DOWN trend", snap_f[snap_f.trend_dir<0])]:
    print(f"\n--- {dname}  (N={len(sub):,}) ---")
    show_surface(sub, "chop", "r2_H", f"D primary: r2_H x chop | {dname} (base {sub.cont_bin.mean():.3f})")

In [ ]:
# Additional surfaces
show_surface(snap_f, "chop", "mag_H",  "D: magnitude x chop (pooled)")
show_surface(snap_f, "mag_H", "r2_H",  "D: cleanliness x magnitude (pooled)")

# Facet by trend-age band -> is loyalty a survival effect?
snap_f["age_band"] = qbin(snap_f["trend_age"], 3)
for ab, g in snap_f.groupby("age_band"):
    print(f"\n--- trend_age band {int(ab)} (median age {g.trend_age.median():.0f} min, N={len(g):,}, cont {g.cont_bin.mean():.3f}) ---")
    show_surface(g, "chop", "r2_H", f"D faceted: r2_H x chop | age band {int(ab)}")

**Where does it glow?** If the bright region is the extreme strong-trend/low-chop
**corner**, loyalty is a momentum corner. If the peak is **interior** (moderately
strong, moderately clean), the regime is a balance, not an extreme. The age facets
reveal whether continuation is highest early in a leg (loyalty rises early, decays
late) — a survival/hazard reading.

## E. Robustness & validity

* Overlap correction (naive-vs-corrected CI gap shown once).
* Walk-forward stability of trend-only separation, the +zigzag increment, and the high region.
* Per-asset consistency.
* Surrogate null (block-bootstrapped returns, block >= 240): rebuild features+outcomes; the lift must collapse.
* Parameter sensitivity (clip, delta set, H, tau, barrier, clipped vs unclipped slopes).

In [ ]:
# E1 naive vs overlap-corrected CI gap (shown once).
# headline snap_f is ALREADY stride-240 (non-overlapping). Build a dense (stride 60) view
# from one asset for the contrast, then compare naive binomial vs block-bootstrap CI.
def build_dense(asset):
    cfg2 = dict(CFG); cfg2["stride"] = CFG["stride_dense"]
    df = load_asset_vwap(asset, CFG["start"], CFG["end"])
    s = build_asset_snapshots(asset, df, cfg2)
    s = finalize(s, cfg2)
    return s

dense = build_dense(CFG["assets"][0])
print(f"dense ({CFG['stride_dense']}-stride) snapshots for {CFG['assets'][0]}: {len(dense):,}")
k_, n_ = dense.cont_bin.sum(), len(dense)
naive = wilson_ci(k_, n_)
mean_bb, blo, bhi = block_bootstrap_mean(dense.cont_bin.values, CFG["block_len"], n_boot=400)
print(f"  cont rate = {k_/n_:.4f}")
print(f"  NAIVE binomial 95% CI       : [{naive[0]:.4f}, {naive[1]:.4f}]  width {naive[1]-naive[0]:.4f}")
print(f"  BLOCK-BOOTSTRAP 95% CI (>=240): [{blo:.4f}, {bhi:.4f}]  width {bhi-blo:.4f}")
print("  -> the naive CI is the mirage; block-bootstrap is wider because overlapping rows are not independent.")

In [ ]:
# E2 walk-forward stability: trend-only separation + zigzag increment per fold + high-region.
ts = snap_f.ts.to_numpy(); n = len(snap_f)
bounds = np.linspace(0, n, CFG["wf_folds"]+1).astype(int)
fold_rows = []
for f in range(CFG["wf_folds"]):
    g = snap_f.iloc[bounds[f]:bounds[f+1]]
    if len(g) < 100: continue
    auc = roc_auc_score(g.cont_bin, g.trend_strength) if g.cont_bin.nunique()>1 else np.nan
    rho_chop = conditional_stat(g.assign(ts_band=qbin(g.trend_strength, CFG["n_qbins"])))
    # high region = top-quintile trend_strength & bottom-tercile chop
    hi = g[(g.trend_strength >= g.trend_strength.quantile(0.8)) & (g.chop <= g.chop.quantile(0.33))]
    fold_rows.append(dict(fold=f, n=len(g),
                          date=str(pd.to_datetime(g.ts.iloc[0], unit="ms").date()),
                          auc_trend=auc, cond_chop_sep=rho_chop,
                          base=g.cont_bin.mean(),
                          hi_region_rate=hi.cont_bin.mean() if len(hi)>20 else np.nan,
                          hi_lift=(hi.cont_bin.mean()-g.cont_bin.mean()) if len(hi)>20 else np.nan))
wf_stab = pd.DataFrame(fold_rows)
print("E2 walk-forward stability"); print(wf_stab.to_string(index=False))
fig = go.Figure()
fig.add_trace(go.Scatter(x=wf_stab.fold, y=wf_stab.auc_trend, mode="lines+markers", name="AUC trend"))
fig.add_trace(go.Scatter(x=wf_stab.fold, y=wf_stab.hi_lift, mode="lines+markers", name="high-region lift"))
fig.add_hline(y=0.5, line_dash="dot"); fig.add_hline(y=0.0, line_dash="dot")
fig.update_layout(title="E2 lift over time (does the regime persist?)", height=380, width=820)
fig.show()

In [ ]:
# E3 per-asset consistency
rows = []
for a, g in snap_f.groupby("asset"):
    if len(g) < 100: continue
    auc = roc_auc_score(g.cont_bin, g.trend_strength) if g.cont_bin.nunique()>1 else np.nan
    rho = spearman(g.r2_H, g.cont_mag)
    rows.append(dict(asset=a, n=len(g), base=g.cont_bin.mean(),
                     auc_trend=auc, rho_r2_cont=rho))
print("E3 per-asset consistency"); print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# E4 surrogate null: block-bootstrap the minute returns (block>=240), rebuild a synthetic
# vwap path, rebuild features+outcomes, recompute trend-only AUC. Lift must collapse.
def surrogate_auc(asset, n_sur, block, seed=0):
    rng = np.random.default_rng(seed)
    df = load_asset_vwap(asset, CFG["start"], CFG["end"])
    vw = df["vwap"].to_numpy(np.float64)
    ret = np.diff(np.log(vw))
    aucs = []
    for s in range(n_sur):
        nb_blocks = int(np.ceil(len(ret)/block))
        starts = rng.integers(0, len(ret), size=nb_blocks)
        idx = (starts[:,None] + np.arange(block)[None,:]).ravel() % len(ret)
        sret = ret[idx[:len(ret)]]
        svw = np.exp(np.concatenate([[np.log(vw[0])], np.cumsum(sret)]))
        sdf = pd.DataFrame({"ts": df.ts.values[:len(svw)], "vwap": svw})
        cfg2 = dict(CFG); cfg2["stride"] = 240; cfg2["max_snaps_per_asset"] = 1500
        ss = build_asset_snapshots(asset+"_sur", sdf, cfg2)
        if ss.empty: continue
        ss = finalize(ss, cfg2)
        ss["trend_strength"] = ss.r2_H*ss.mag_H
        if ss.cont_bin.nunique()>1:
            aucs.append(roc_auc_score(ss.cont_bin, ss.trend_strength))
    return np.array(aucs)

asset0 = CFG["assets"][0]
real_auc = roc_auc_score(snap_f[snap_f.asset==asset0].cont_bin,
                         snap_f[snap_f.asset==asset0].trend_strength)
sur = surrogate_auc(asset0, n_sur=30, block=CFG["block_len"])   # 30 keeps runtime sane; raise for final
print(f"E4 surrogate null ({asset0})")
print(f"  REAL trend-only AUC      = {real_auc:.4f}")
print(f"  surrogate AUC mean+-std  = {sur.mean():.4f} +- {sur.std():.4f}  (95th {np.percentile(sur,95):.4f})")
p = (np.sum(sur >= real_auc)+1)/(len(sur)+1)
print(f"  p(real <= surrogate)     = {p:.4f}  ({'beats surrogate null' if p<0.05 else 'does NOT beat surrogate'})")
fig = go.Figure(); fig.add_trace(go.Histogram(x=sur, name="surrogate AUC"))
fig.add_vline(x=real_auc, line_color="red", annotation_text="real")
fig.update_layout(title="E4 real vs surrogate trend-only AUC", height=360, width=720); fig.show()

In [ ]:
# E5 parameter sensitivity: H, slope kind (clipped vs unclipped), tau, barrier.
sens = []
for H in CFG["reg_windows"]:
    for kind in ["c","u"]:
        sf = finalize(snap, dict(CFG, chop_scalar_delta=0.5), H=H, slope_kind=kind)
        auc = roc_auc_score(sf.cont_bin, (sf.r2_H*sf.mag_H)) if sf.cont_bin.nunique()>1 else np.nan
        rho = spearman(sf.r2_H, sf.cont_mag)
        cond = conditional_stat(sf.assign(ts_band=qbin(sf.r2_H*sf.mag_H, CFG["n_qbins"]),
                                          trend_strength=sf.r2_H*sf.mag_H))
        sens.append(dict(H=H, slope=kind, base=sf.cont_bin.mean(),
                         auc_trend=auc, rho_r2=rho, cond_chop=cond))
sens = pd.DataFrame(sens)
print("E5 parameter sensitivity (H x clipped/unclipped slopes)")
print(sens.to_string(index=False))
print("\n  clipped-vs-unclipped: compare auc_trend within each H to see which slope predicts better.")

## F. (Optional) Decision value — exploratory

Toy gate: take the trend direction **only when in the high-continuation region**
vs **always**. Compare OOS forward-return distribution and a crude Sharpe. Clearly
exploratory — a regime that matters for one strategy can be noise for another.

In [ ]:
# Walk-forward: define high region on past fold, trade next fold. Net = trend_dir * fwd_ret_240.
ts = snap_f.ts.to_numpy(); n = len(snap_f); bnd = np.linspace(0,n,CFG["wf_folds"]+1).astype(int)
gated_ret, always_ret = [], []
for f in range(1, CFG["wf_folds"]):
    tr = snap_f.iloc[:bnd[f]]; te = snap_f.iloc[bnd[f]:bnd[f+1]]
    if len(tr)<200 or len(te)<50: continue
    th_str  = tr.trend_strength.quantile(0.8)
    th_chop = tr.chop.quantile(0.33)
    gate = (te.trend_strength >= th_str) & (te.chop <= th_chop)
    gated_ret.append(te.loc[gate, "cont_mag"].values)
    always_ret.append(te["cont_mag"].values)
g = np.concatenate(gated_ret) if gated_ret else np.array([])
al = np.concatenate(always_ret)
def sharpe(x): return x.mean()/x.std()*np.sqrt(252*6.5*60/240) if len(x)>1 and x.std()>0 else np.nan
print("F exploratory decision value (cont_mag = trend_dir * fwd_ret_240, OOS)")
print(f"  ALWAYS-on : n={len(al):,}  mean={al.mean():+.5f}  win%={(al>0).mean():.3f}  crudeSharpe={sharpe(al):.3f}")
print(f"  GATED     : n={len(g):,}  mean={g.mean():+.5f}  win%={(g>0).mean():.3f}  crudeSharpe={sharpe(g):.3f}")
fig = go.Figure()
fig.add_trace(go.Histogram(x=al, name="always", opacity=0.6, histnorm="probability density"))
if len(g): fig.add_trace(go.Histogram(x=g, name="gated", opacity=0.6, histnorm="probability density"))
fig.update_layout(barmode="overlay", title="F OOS forward-return (gated vs always)", height=360, width=720); fig.show()

## G. Verdict cell — computed, not asserted

This cell collects the headline numbers and prints the verdict per the three
options in section 0, the boundaries of the validated region, its stability across
time and assets, whether it beats the surrogate null, and a **kill criterion**.

In [ ]:
def fmt(x): return "n/a" if x is None or (isinstance(x,float) and not np.isfinite(x)) else f"{x:.4f}"

# pull the key statistics computed above
trend_auc      = auc_trend
trend_rho      = rho_r2
cond_chop_pval = pval                      # permutation null p for chop|trend
cond_chop_obs  = obs
incr_auc       = wf_delta.get("auc", float("nan"))
incr_r2        = wf_delta.get("r2", float("nan"))
surr_p         = globals().get("p", float("nan"))   # surrogate-null p-value from E4
wf_auc_min     = wf_stab.auc_trend.min() if not wf_stab.empty else np.nan
wf_auc_mean    = wf_stab.auc_trend.mean() if not wf_stab.empty else np.nan
per_asset_min  = pd.DataFrame(rows).auc_trend.min() if rows else np.nan

# decision thresholds
TREND_REAL   = (trend_auc > 0.53) and (surr_p < 0.05) and (wf_auc_mean > 0.51)
CHOP_ADDS    = (cond_chop_pval < 0.05) and (incr_auc > 0.002) and (cmi > 0.001)

if TREND_REAL and CHOP_ADDS:
    verdict = "1. VALIDATED *WITH* ZIGZAG"
elif TREND_REAL and not CHOP_ADDS:
    verdict = "2. VALIDATED *WITHOUT* ZIGZAG (chop adds nothing once trend is fixed)"
else:
    verdict = "3. NOT VALIDATED (neither separates the future beyond the null)"

print("="*78)
print("VERDICT:", verdict)
print("="*78)
print(f"Base continuation rate (stride-240, N={N:,})   : {fmt(base_rate)}")
print()
print("Trend-only separation")
print(f"  AUC(trend_strength -> cont_bin)              : {fmt(trend_auc)}")
print(f"  Spearman(r2_H, cont_mag)                     : {fmt(trend_rho)}")
print(f"  walk-forward AUC  mean / min                 : {fmt(wf_auc_mean)} / {fmt(wf_auc_min)}")
print(f"  per-asset AUC min                            : {fmt(per_asset_min)}")
print(f"  beats surrogate null (p<0.05)                : {surr_p < 0.05}  (p={fmt(surr_p)})")
print()
print("Zigzag increment (holding trend fixed)")
print(f"  permutation-null p (chop|trend)              : {fmt(cond_chop_pval)}")
print(f"  conditional MI  I(cont;chop|trend) [nats]    : {fmt(cmi)}")
print(f"  OOS delta AUC  (trend+chop - trend)          : {fmt(incr_auc)}")
print(f"  OOS delta R2   (trend+chop - trend)          : {fmt(incr_r2)}")
print()
print("Boundaries of the validated region (where the surface glows):")
hi = snap_f[(snap_f.trend_strength>=snap_f.trend_strength.quantile(0.8)) &
            (snap_f.chop<=snap_f.chop.quantile(0.33))]
print(f"  high region  = top-quintile trend_strength AND bottom-tercile chop")
print(f"  high-region continuation rate                : {fmt(hi.cont_bin.mean())} "
      f"(lift {fmt(hi.cont_bin.mean()-base_rate)} vs base)")
print()
# kill criterion: rolling forward-separation floor
wf_std = wf_stab.auc_trend.std() if not wf_stab.empty else float("nan")
kill = max(0.505, (wf_auc_mean - 2*wf_std)) if np.isfinite(wf_auc_mean) and np.isfinite(wf_std) else 0.505
print(f"KILL CRITERION: declare the regime dead if the rolling walk-forward")
print(f"  trend-only AUC falls below {fmt(kill)} for two consecutive folds")
print(f"  (mean {fmt(wf_auc_mean)}, std {fmt(wf_std)}); treat as a candidate with a shelf life.")

### Written conclusion

Fill from the computed numbers above. The structure of a defensible conclusion:

- **Verdict** (1/2/3) and the AUC/rho/increment numbers behind it.
- **Trend alone**: does `trend_strength -> cont_bin` clear AUC ~0.53, survive the
  surrogate null, and stay positive across all walk-forward folds and all 7 assets?
- **Zigzag's marginal value**: once trend is fixed, does chop still separate
  continuation (permutation p < 0.05, positive conditional MI, positive OOS delta-AUC)?
  If flat, **say plainly** that chop is a momentum signal in a costume (verdict 2).
- **Region boundaries**: corner vs interior peak, symmetry up/down, the trend-age
  survival reading.
- **Stability**: lift-over-time plot, per-asset table.
- **Kill criterion**: the rolling forward-AUC floor below which the regime is dead.

Someone may disagree only by disputing these numbers — overlap, leak, the null,
out-of-sample, the complement, and "does trend alone already do it" are all
addressed above.